# Uncertainty at the Threshold:  Sampling Error in Policy-Relevant Composite Indicators

In [ ]:
import os
import sys

REPO_ROOT = os.path.dirname(os.getcwd())
FUNCTIONS_DIR = os.path.join(REPO_ROOT, "1_code", "functions")

if FUNCTIONS_DIR not in sys.path:
    sys.path.append(FUNCTIONS_DIR)

print(REPO_ROOT)
print(FUNCTIONS_DIR)
print(os.path.exists(FUNCTIONS_DIR))

#### Loading Demo Data

In [ ]:
import os
import pandas as pd

DEMO_DIR = os.path.join(
    REPO_ROOT,
    "0_data",
    "input",
    "demo",
    "synthetic_2022"
)

def load_synthetic_replicates(demo_dir=DEMO_DIR):
    dataframes = {}
    table_names = []

    for filename in sorted(os.listdir(demo_dir)):
        if not filename.endswith(".csv"):
            continue

        table_name = filename.replace(".csv", "")
        table_names.append(table_name)

        dataframes[table_name] = pd.read_csv(
            os.path.join(demo_dir, filename),
            dtype={"GEOID": str}
        )

    return dataframes, table_names


dataframes, table_names = load_synthetic_replicates()

print(f"Loaded {len(table_names)} synthetic ACS VRT tables.")
print(table_names)

#### Building indicator
- sum of multiple fields to derive indicator
- for Estimate and all variance replicate columns
- final tables each table estimate and 80 replicates of indicator 
- directory: calculated_indicator_tables

In [ ]:
from replicate_indicators import calculate_indicators
import pandas as pd
from IPython.display import display
import os
from csv_file import save_to_csv, read_from_csv


df_indicators = calculate_indicators(dataframes)

# Save the indicators to a csv file
save_to_csv(df_indicators, 'output/demo', '2022_df_indicators.csv')

#### Calculate denominator indicators 
- TOTPOP total population
- TOTHH total households
- TOTHU total housing units

In [ ]:
import pandas as pd
from replicate_indicators import calculate_denominators
import os 
from csv_file import save_to_csv, read_from_csv

# Call the function and create the dataset
df_denominators = calculate_denominators(dataframes)

save_to_csv(df_denominators, 'output/demo', '2022_df_denominators.csv')   


#### Calculate Nominal Index and MOE of Index
- NOMINAL index 

In [ ]:
import os
import pandas as pd
from calculate_denomination import estimate_denomination
from nominal_index import construct_nominal_index
from csv_file import save_to_csv, read_from_csv

# read input datasets
df_indicators = read_from_csv('2022_df_indicators.csv', 'output/demo')
df_denominators = read_from_csv('2022_df_denominators.csv', 'output/demo')

# denominate indicators based on Estimate of indicator and denominator
df_variables = estimate_denomination(df_indicators, df_denominators)

# Get the initial number of rows
initial_rows = len(df_variables)

# Remove rows where the 'POP' column is 0
df_variables = df_variables[df_variables['POP'] != 0].copy()

# Get the final number of rows
final_rows = len(df_variables)

# Calculate and print the number of rows removed
rows_removed = initial_rows - final_rows
print(f"Number of rows removed: {rows_removed}")

# Construct nominal index percentile rank normalization
nominal_index = construct_nominal_index(df_variables, 'pct', True)

# save nominal index to csv
save_to_csv(nominal_index, 'output/demo', 'nominal_index_pct.csv')

#### Replicate Index Construction 

In [ ]:
from replicate_indices import construct_replicate_indices
from csv_file import save_to_csv, read_from_csv
from moe_stability import class_stability

# Read nominal index
df_nominal_index_pct = read_from_csv(
    'nominal_index_pct.csv',
    'output/demo'
)

# Construct 80 replicate indices
var_rep_indices_pct = construct_replicate_indices(
    df_nominal_index_pct,
    'pct',
    True
)

# Save replicate indices
save_to_csv(
    var_rep_indices_pct,
    'output/demo',
    'var_rep_indices_pct.csv'
)

# Calculate classification stability / MOE
df_index_moe = class_stability(var_rep_indices_pct)

# Save result needed by later cells
save_to_csv(
    df_index_moe,
    'output/demo',
    'vrt_indices_moe_stability.csv'
)

display(df_index_moe.head())

### Figure 2 XSP-based classification uncertainty across a composite social vulnerability index

In [ ]:
from uncertainty_plots import plot_classification_uncertainty
from csv_file import read_from_csv
import matplotlib.pyplot as plt

# Example usage
df = read_from_csv('vrt_indices_moe_stability.csv', 'output/demo')

fig, ax = plot_classification_uncertainty(
    df,
    nominal_col='NOMINAL',
    var_rep_prefix='Index_Rep',
    n_replicates=80,
    fig_size=(7.2, 7.2),
    reference_line=0.75,
    scale_to_100=True,
    add_designation_regions=True
)

plt.savefig('../2_plots/model_comparison.png', dpi=300, bbox_inches='tight')
plt.show()

### Implementation of Designation Under Uncertainty

Approach I
	- Top ten thousand tract
	- narrative discussion around it
	- then practicle example
	- tracts confidence 
	- second number how many tracts stay the same for confidence interval 
	- giving practical path forward
- Approach 2
	- above 75
		- count how often tracts are above 75
	- how do we make a cut?
		- 8000 tracts above 75
		- then sort by number of designation
		- pick top 8000 again 

In [ ]:
import pandas as pd
from collections import Counter
from csv_file import read_from_csv

# Load data
df = read_from_csv('vrt_indices_moe_stability.csv', 'output/demo')

# Parameters
nominal_col = 'NOMINAL'
var_rep_prefix = 'Index_Rep'
n_replicates = 80
top_k = 100

# Ensure index is GEOID
df = df.set_index('GEOID')

# Create full list of simulation columns
all_sim_cols = [f'{var_rep_prefix}{i+1}' for i in range(n_replicates)] + [nominal_col]

# Initialize counting column
df['top_100'] = 0

# Dictionary to store sets
top_100_sets = {}

# Loop through all simulations and increment count
for col in all_sim_cols:
    if col in df.columns:
        threshold = df[col].nlargest(top_k).min()
        top_ids = df.nlargest(top_k, col).index
        df.loc[top_ids, 'top_100'] += 1
        top_100_sets[col] = set(top_ids)

        if col == nominal_col:
            nominal_threshold = threshold
    else:
        print(f"Warning: Column '{col}' not found.")

# Build final list based on total appearances
final_top_ids = df['top_100'].nlargest(top_k).index
nominal_top_ids = top_100_sets[nominal_col]

# Identify new tracts not in original NOMINAL top 100
newly_included_ids = set(final_top_ids) - nominal_top_ids

# Extract NOMINAL values
lowest_nominal_in_nominal_top = df.loc[list(nominal_top_ids), nominal_col].min()

if newly_included_ids:
    lowest_nominal_in_new = df.loc[list(newly_included_ids), nominal_col].min()
else:
    lowest_nominal_in_new = None

# Output
print("\n---- FINAL SUMMARY ----")
print(f"Total number of tracts: {len(df)}")
print(f"NOMINAL threshold to be in top {top_k}: {nominal_threshold:.5f}")
print(f"Lowest NOMINAL value in original top {top_k}: {lowest_nominal_in_nominal_top:.5f}")

if lowest_nominal_in_new is not None:
    print(f"Lowest NOMINAL value among newly included tracts: {lowest_nominal_in_new:.5f}")
else:
    print("No newly included tracts.")

print(f"Total newly included tracts: {len(newly_included_ids)}")
print(f"Final top {top_k} (based on frequency): {len(final_top_ids)}")

In [ ]:
import pandas as pd
import numpy as np
from csv_file import read_from_csv
from collections import Counter
from IPython.display import display

# Load data
df = read_from_csv('vrt_indices_moe_stability.csv', 'output/demo')

# Parameters
nominal_col = 'NOMINAL'
var_rep_prefix = 'Index_Rep'
n_replicates = 80
threshold = 0.75

# Set GEOID as index
df = df.set_index('GEOID')

# Create list of replicate columns
rep_cols = [f'{var_rep_prefix}{i+1}' for i in range(n_replicates)]

# Step 1: Get NOMINAL designated tracts
num_nominal_designated = (df[nominal_col] >= threshold).sum()
nominal_designated_ids = set(df[df[nominal_col] >= threshold].index)

# Step 2: Binary matrix of replicate designations
binary_designation = pd.DataFrame({
    col: (df[col] >= threshold).astype(int)
    for col in rep_cols
}, index=df.index)

# Step 3: Count how many times each tract is designated
df['replicate_designation_count'] = binary_designation.sum(axis=1)

# Step 4: Keep tracts designated at least once
designated_at_least_once = df[df['replicate_designation_count'] > 0]

# Step 5: Top replicate-based tracts (same number as NOMINAL)
top_by_replicates = designated_at_least_once.nlargest(
    num_nominal_designated,
    'replicate_designation_count'
)
top_by_replicates_ids = set(top_by_replicates.index)

# Step 6: Compare overlaps and differences
intersection = nominal_designated_ids & top_by_replicates_ids
only_nominal = nominal_designated_ids - top_by_replicates_ids
only_replicates = top_by_replicates_ids - nominal_designated_ids

print("\n---- SECOND TASK (Using GEOID) ----")
print(f"Number of tracts with NOMINAL >= 0.75: {num_nominal_designated}")
print(f"Number of tracts designated at least once in replicates: {len(designated_at_least_once)}")
print(f"Overlap with replicate-based top count: {len(intersection)}")
print(f"Only in NOMINAL, not in replicate-based top: {len(only_nominal)}")
print(f"Only in replicate-based top, not in NOMINAL: {len(only_replicates)}")

# Step 7: Create DataFrame of mismatches and display
df_mismatch = df.loc[list(only_nominal | only_replicates)].copy()
df_mismatch['in_nominal'] = df_mismatch.index.isin(nominal_designated_ids)
df_mismatch['in_replicates'] = df_mismatch.index.isin(top_by_replicates_ids)

print("\n--- Mismatched Tracts (head) ---")
display(df_mismatch.head())

# Step 8: Print lowest NOMINAL values among replicate-based tracts not in NOMINAL
newly_designated_ids = only_replicates

if newly_designated_ids:
    lowest_nominal_among_new = df.loc[
        list(newly_designated_ids),
        nominal_col
    ].sort_values().head(10)

    print("\n--- Lowest NOMINAL values among replicate-designated but not nominal-designated tracts ---")
    print(lowest_nominal_among_new)
else:
    print("\nNo replicate-designated tracts found that were not in NOMINAL.")

In [ ]:
from csv_file import read_from_csv
import matplotlib.pyplot as plt
from moe_stability import analyze_moe_regions

# Example usage
df = read_from_csv('vrt_indices_moe_stability.csv', 'output/demo')
df_denominators = read_from_csv('2022_df_denominators.csv', 'output/demo')

region_stats = analyze_moe_regions(df, reference_line=0.75)

# Merge df with denominator data (which includes TOTPOP)
merged = df.merge(df_denominators[['GEOID', 'E_TOTPOP']], on='GEOID', how='left')

# Define uncertainty bands (scaled 0–1)
pfn = merged[(merged['NOMINAL'] >= 0.63) & (merged['NOMINAL'] < 0.75)]  # potential false negatives
pfp = merged[(merged['NOMINAL'] >= 0.75) & (merged['NOMINAL'] < 0.82)]  # potential false positives
uncertainty_zone = merged[(merged['NOMINAL'] >= 0.63) & (merged['NOMINAL'] < 0.82)]

# Count and sum population
print(f"Potential False Negatives (63–75): {len(pfn):,} tracts, {pfn['E_TOTPOP'].sum():,.0f} people")
print(f"Potential False Positives (75–82): {len(pfp):,} tracts, {pfp['E_TOTPOP'].sum():,.0f} people")
print(f"Total Uncertainty Zone (63–82): {len(uncertainty_zone):,} tracts, {uncertainty_zone['E_TOTPOP'].sum():,.0f} people")

### Spatial Analysis 

In [ ]:
import pandas as pd
from csv_file import read_from_csv
from distance_to_designation import calculate_steps
from io_utils import write_parquet
import geopandas as gpd
import numpy as np
import os

# read input dataset
df = read_from_csv('var_rep_indices_pct.csv', 'output/demo')
df['GEOID'] = df['GEOID'].astype(str)

# Count the number of designations
df['nom_desig'] = (df['NOMINAL'] >= 0.99).astype(int)

# VRT designation columns
cols_to_check = ['NOMINAL'] + [f'Index_Rep{i}' for i in range(1, 81)]

# Count the number of designations across all replicates
df['count_design'] = (df[cols_to_check] >= 0.99).sum(axis=1)
df['count_design'] = df['count_design'].apply(lambda x: x if x >= 1 else None)

# Load synthetic tract geometries
gdf = gpd.read_file(
    os.path.join(
        REPO_ROOT,
        '0_data',
        'input',
        'demo',
        'synthetic_grid.gpkg'
    )
)

gdf['GEOID'] = gdf['GEOID'].astype(str)

# Merge index data with synthetic geometries
merged_gdf = gdf.merge(df, on='GEOID', how='left')
merged_gdf = merged_gdf.reset_index(drop=True)

# Find tracts which are designated through replicate indices but not by nominal estimate index (top 1%)
filtered_tracts = merged_gdf[
    (merged_gdf['count_design'] >= 1) &
    (merged_gdf['nom_desig'] != 1)
]

print(f"Replicate-only designated demo tracts: {len(filtered_tracts)}")

# Calculate the distance number of tracts to pass through to the closest nominal estimate designation
merged_gdf = calculate_steps(merged_gdf, filtered_tracts)
merged_gdf = merged_gdf.rename(columns={'steps_to_nom_desig': 'moe_distance'})

# Random sampling for comparison
n_random_samples = 2000

# Exclude already designated tracts from random sampling
non_designated_tracts = merged_gdf[merged_gdf['nom_desig'] != 1]
print(f"Non-designated demo tracts available for random sampling: {len(non_designated_tracts)}")

# Ensure we don't sample more than available
actual_sample_size = min(n_random_samples, len(non_designated_tracts))

# Reproducible random sample
rng = np.random.default_rng(42)

random_sample_indices = rng.choice(
    non_designated_tracts.index,
    size=actual_sample_size,
    replace=False
)

# Calculate distances for random sample
print(f"Calculating distances for {actual_sample_size} random demo tracts...")

random_sample_tracts = merged_gdf.loc[random_sample_indices]

random_gdf = calculate_steps(merged_gdf, random_sample_tracts)
random_gdf = random_gdf.rename(columns={'steps_to_nom_desig': 'random_distance'})

write_parquet(
    random_gdf,
    'output/demo',
    'moe_random_distance.parquet'
)

### Figure 4 Spatial clustering of census tracts with classification uncertainty

In [ ]:
baseline_color = "#7f8c8d"
spatial_moe_color = "#bd6b57"
random_moe_color = "#5b8c8a"

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from io_utils import read_parquet
import matplotlib.patches as patches
from matplotlib.patches import Circle
from collections import Counter
from plot_distance import create_cumulative_plot, create_distribution_plot, create_spatial_network_plot, create_panel_plot, export_html_to_tiff

# Set style
plt.style.use('default')
sns.set_palette("husl")

data_path = 'output/demo/moe_random_distance.parquet'

# Example usage:
fig1 = create_spatial_network_plot(data_path)
fig2 = create_distribution_plot(data_path) 
fig3 = create_cumulative_plot(data_path)
fig4 = create_panel_plot(data_path)

html_path = "../2_plots/panel_plot.html"
tiff_path = "../2_plots/panel_plot.tiff"

export_html_to_tiff(html_path, tiff_path, width=1600, height=1200, dpi=600)